# UPAR 2027 – Track 2 (Attribute-based Person Retrieval) – training on Colab
**Runtime → Change runtime type → T4 GPU**, then run the cells **one by one** from top to bottom.

The `UPAR-Challenge-2027-main` folder can sit anywhere in your Google Drive (any folder name, any capitalisation):
the first cell finds it. Everything heavy (cache, checkpoints, final zips) is saved in a `work` folder next to it,
so after a disconnect just re-run the cells: finished steps are skipped and training resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, glob
# find the uploaded code anywhere in Drive (folder names are case-sensitive, so don't guess them)
hits = sorted(h for h in glob.glob('/content/drive/MyDrive/**/solution/train.py', recursive=True) if '/work/' not in h)
assert hits, 'solution/train.py not found in Drive: is the upload of UPAR-Challenge-2027-main finished?'
SRC = os.path.dirname(os.path.dirname(hits[0]))      # .../UPAR-Challenge-2027-main
DRIVE = os.path.dirname(SRC)                          # the folder that contains it
WORK = f'{DRIVE}/work'
os.makedirs(WORK, exist_ok=True)
print('code :', SRC)
print('work :', WORK)
!rsync -a --exclude data/Market1501 --exclude data/PA100k --exclude data/PETA "{SRC}/" /content/upar/
%cd /content/upar/solution
!ls
!pip -q install gdown open_clip_torch

## 1. Data: download (~15 min) and build the 256x128 cache (~10 min). Skipped if the cache is already on Drive.

In [ ]:
%cd /content/upar
import os
if not os.path.exists(f'{WORK}/cache/val_labels.npy'):
    !python download_datasets.py --data-dir ./data
    !cd solution && python prepare_cache.py --data-root ../data --out "{WORK}/cache" --size 256 128
!ls -lh "{WORK}/cache"

## 2. CLIP ViT-B/16 weights (OpenAI, allowed: generic foundation model, not trained for PAR)

In [ ]:
%cd /content/upar/solution
if not os.path.exists(f'{WORK}/weights/clip_ViT-B-16.pt'):
    !python fetch_clip.py --arch ViT-B-16 --out-dir "{WORK}/weights"
!ls -lh "{WORK}/weights"

## 3. Development run: train on *train*, validate on *val* every epoch (~2 h on T4)
This gives the numbers for the report and tunes the retrieval scoring.
Validation mADM peaks after **3 epochs** (0.737) and then declines, so `model_best.pt` is the 3-epoch checkpoint.

In [ ]:
%cd /content/upar/solution
!python train.py --cache {WORK}/cache --train-splits train --val-split val    --queries ../data/annotations/task2/val/queries.csv    --clip-weights {WORK}/weights/clip_ViT-B-16.pt --epochs 10 --bs 128 --out {WORK}/runs/dev

In [ ]:
%cd /content/upar/solution
!python evaluate.py --ckpt {WORK}/runs/dev/model_best.pt --cache {WORK}/cache --split val    --queries ../data/annotations/task2/val/queries.csv --out {WORK}/runs/dev

## 4. Quick first submission (optional but recommended): zip the dev model now
Upload `upar_t2_dev.zip` on Codabench → Track 2 → My Submissions. This proves the pipeline works on their servers
while the final model trains.

In [ ]:
%cd /content/upar/solution
!python build_submission.py --ckpt {WORK}/runs/dev/model_best.pt --scoring {WORK}/runs/dev/scoring.json --out {WORK}/upar_t2_dev.zip

## 5. Final run: train + val (allowed by the rules) for 3 epochs, the length selected on validation (~45 min on T4)

In [ ]:
%cd /content/upar/solution
!python train.py --cache {WORK}/cache --train-splits train val    --clip-weights {WORK}/weights/clip_ViT-B-16.pt --epochs 3 --bs 128 --out {WORK}/runs/final3

In [ ]:
%cd /content/upar/solution
# final submission: ensemble of the train+val model and the validation-selected model
!python build_submission.py --ckpt {WORK}/runs/final3/model_ema.pt {WORK}/runs/dev/model_best.pt \n   --scoring {WORK}/runs/dev/scoring.json --out {WORK}/upar_t2_ensemble.zip

## 6. Report material
`runs/dev/log.jsonl` (per-epoch curves), `runs/dev/results.json` (Track 1 + Track 2 metrics, scoring grid,
per-dataset breakdown). Download them from Drive and send them back for the report.

**Codabench test results:** train only 0.619 mADM · train+val 0.613 · ensemble (final submission) **0.624**, rank 32.

In [ ]:
%cd /content/upar/solution
import json
for l in open(f'{WORK}/runs/dev/log.jsonl'): print(l.strip())
r = json.load(open(f'{WORK}/runs/dev/results.json'))
print(json.dumps({k: r[k] for k in ['track1', 'track2_l1_baseline_scoring', 'track2_default', 'track2_best', 'per_dataset']}, indent=1))